# M4L3 - Graphing of RNA-Seq Results in `R`
In this notebook, we’ll move beyond data frames, functions, and packages, and focus on graphical visualizations of RNA‑Seq results. This is using an example workflow going from raw counts to producing results on a small sample set, similar to what we had done last week in our use of the pseudo-alignment pipeline. Don't focus too much on the use of `DESeq2`, we will cover this and results interpretation within the practices. This lab is made to demonstrate the use of `R` packages for graphing DE results and to be used as a reference for our Exercise this week. 

**NOTE:** We are going to be using `R` within this notebook within our `Module 3 - R` Kernel. If you have not completed this or had issues in M3P2, use the course-wide Module 3 environment (see `environment/module_3_rnaseq_salmon.yml`) or the Binder environment (`binder/environment.yml`), which have every package this notebook needs.

## In this Lab, we’ll cover:

- Briefly setting up `DESeq2`
- Principal Component Analysis (PCA)
- Volcano plots (differential expression)
- MA plots and plots of counts for single genes
- Heat maps (expression across samples)

We will use a dataset from [“RNA-Seq Transcriptome Profiling Identifies CRISPLD2 as a Glucocorticoid Responsive Gene that Modulates Cytokine Function in Airway Smooth Muscle Cells.”](https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0099625). This data was downloaded from [GSE52778](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE52778), and was processed using `kallisto` and `tximport`. 

This study used RNA-seq to profile gene expression changes in four ASM cell lines treated with dexamethasone, a synthetic glucocorticoid. They found several differentially expressed genes comparing dexamethasone-treated ASM cells to control cells, but focused much of the discussion on a gene called CRISPLD2. This gene encodes a secreted protein known to be involved in lung development, and SNPs in this gene in previous GWAS studies are associated with inhaled corticosteroid resistance and bronchodilator response in asthma patients. They confirmed the upregulated CRISPLD2 mRNA expression with qPCR and increased protein expression using Western blotting.

```{admonition} Adapted from
:class: tip
**Source:** Adapted from [Count-Based Differential Expression Analysis of RNA-seq Data](https://4va.github.io/biodatasci/r-rnaseq-airway.html) by Stephen Turner (University of Virginia), part of the 4-VA JMU 2017 Biological Data Science Workshop, [4va/biodatasci](https://github.com/4va/biodatasci), licensed [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/). The countData/colData figure and the files `airway_scaledcounts.csv`, `airway_metadata.csv` and `annotables_grch38.csv` come unchanged from the same repository.

**Changes:** Reduced to the data import, DESeq2 and plotting sections and rearranged as a Jupyter notebook; the background, "poor man's DGE", pathway-analysis and homework sections and the exercises were left out; the MA- and volcano-plot exercises were turned into worked `ggplot2` code with new line-by-line explanations; the heat map was redone with `pheatmap` (top 50 genes, sample annotations and colours) instead of `NMF::aheatmap`; a count/significance summary and a short note on gray versus missing points were added; file paths now point to the repository `data/airway/` folder and a folder screenshot was removed. Much of the remaining explanatory text follows the original closely.

**License of this notebook:** This notebook and its accompanying data files (`data/airway/airway_scaledcounts.csv`, `data/airway/airway_metadata.csv`, the downloaded `annotables_grch38.csv`, and the figure in `images/`) are distributed under [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/), not under the repository's all-rights-reserved notice. See [LICENSE](../../../LICENSE) and [ACKNOWLEDGMENTS.md](../../../ACKNOWLEDGMENTS.md).
```

In [ ]:
#Import libraries: 
library(tidyverse)
library("DESeq2")
library(dplyr)
library(ggplot2)
library(pheatmap)

## Data structure

We will use three CSV files in the repository folder `data/airway/`. `airway_scaledcounts.csv` and `airway_metadata.csv` are committed; `annotables_grch38.csv` (7.7 MB) is downloaded by `bash data/fetch_data.sh airway_tables` (also part of `bash data/fetch_data.sh small`, which Binder runs).

The data within `airway_scaledcounts.csv` and `airway_metadata.csv` are structured as follows: 

![countData and colData schematic](images/M4L3_Graphing_RNA_Seq_Results_28fd7d8a.png)

*Figure: `countdatacoldata.png` from [4va/biodatasci](https://github.com/4va/biodatasci) (Stephen Turner), CC BY-NC-SA 4.0, unchanged.*

*Note: this is a generic schematic - our genes are not actually geneA and geneB, and our samples aren’t called ctrl_1, ctrl_2, etc*

We have two tables:

- The “count matrix” (called the countData in `DESeq2`) – where genes are in rows and samples are in columns, and the number in each cell is the number of reads that mapped to exons in that gene for that sample: `airway_scaledcounts.csv`.
- The sample metadata (called the colData in `DESeq2`) – where samples are in rows and metadata about those samples are in columns: `airway_metadata.csv`. It’s called the colData because this table supplies metadata/information about the columns of the countData matrix. Notice that the first column of colData must match the column names of countData (except the first, which is the gene ID column).

In [ ]:
#import the counts, metadata, and annotation
DATA_DIR <- Sys.getenv("DATA_DIR", unset = "../../../data")  # repository data/ folder
AIRWAY_DIR <- file.path(DATA_DIR, "airway")                    # run data/fetch_data.sh airway_tables for the annotation file
mycounts <- read_csv(file.path(AIRWAY_DIR, "airway_scaledcounts.csv"))
metadata <- read_csv(file.path(AIRWAY_DIR, "airway_metadata.csv"))
anno <- read_csv(file.path(AIRWAY_DIR, "annotables_grch38.csv"))

In [ ]:
#look at both of our csv files: 
mycounts

In [ ]:
metadata

In [ ]:
anno

**NOTE: The sample IDs in the metadata sheet (SRR1039508, SRR1039509, etc.) exactly match the column names of the countdata, except for the first column, which contains the Ensembl gene ID. This is important, and we’ll get more strict about it later on.**

Remember, we read in our count data and our metadata using read_csv() which read them in as those “special” dplyr data frames / tbls. We’ll need to convert them back to regular data frames for them to work well with DESeq2.

In [ ]:
mycounts <- as.data.frame(mycounts)
metadata <- as.data.frame(metadata)
head(mycounts)
head(metadata)
head(anno)
class(mycounts)
class(metadata)


Let’s check that the column names of our count data (except the first, which is ensgene) are the same as the IDs from our colData.

In [ ]:
names(mycounts)[-1]
metadata$id
names(mycounts)[-1]==metadata$id
all(names(mycounts)[-1]==metadata$id)

## Running `DESeq2`
Now we create our `dds` object with the `DESeqDataSetFromMatrix()` function from our `DESeq2` package.

In [ ]:
dds <- DESeqDataSetFromMatrix(countData=mycounts, 
                              colData=metadata, 
                              design=~dex, 
                              tidy=TRUE)
dds

Let’s run the DESeq pipeline on the dataset and reassign the resulting object back to the same variable. Before we start, dds is a bare-bones DESeqDataSet. The DESeq() function takes a DESeqDataSet and returns a DESeqDataSet, but with lots of other information filled in (normalization, dispersion estimates, differential expression results, etc). Notice how if we try to access these objects before running the analysis, nothing exists.

In [ ]:
dds <- DESeq(dds)

Since we’ve got a fairly simple design (single factor, two groups, treated versus control) that is similar to the design we had within the OPT3 and Col0 pseudo alignment, we can get results out of the object simply by calling the results() function on the DESeqDataSet that has been run through the pipeline. The help page for ?results and the vignette both have extensive documentation about how to pull out the results for more complicated models (multi-factor experiments, specific contrasts, interaction terms, time courses, etc.).

Note two things:

1. We’re passing the tidy=TRUE argument, which tells DESeq2 to output the results table with rownames as a first column called ‘row.’ If we didn’t do this, the gene names would be stuck in the `row.name`s, and we’d have a hard time filtering or otherwise using that column.
2. This returns a regular old data frame. Try displaying it on the screen by just typing res. You’ll see that it doesn’t print as nicely as the data we read in with `read_csv()`. We can add this “special” attribute to the raw data returned, which just tells R to print it nicely.


In [ ]:
res <- results(dds, tidy=TRUE)
res <- tibble::as_tibble(res)
res

In [ ]:
res <- res %>%
  arrange(padj)   # Sort by adjusted p-value

res <- res %>%
  inner_join(anno, by = c("row" = "ensgene"))

sig_genes <- res %>%
  filter(padj < 0.05)

# Count significant genes
n_sig <- nrow(sig_genes)
n_sig

 Let’s write out the significant results. See the help for ?write_csv, which is part of the readr package (note: this is not the same as write.csv with a dot.). We can continue that pipe and write out the significant results to a file like so (the file `sigresults.csv` is written next to this notebook and is not tracked by git):

In [ ]:
res %>% 
  filter(padj<0.05) %>% 
  write_csv("sigresults.csv")

## Plotting for RNA-Seq results: 
#### Plotting counts

`DESeq2` offers a function called plotCounts() that takes a DESeqDataSet that has been run through the pipeline, the name of a gene, and the name of the variable in the colData that you’re interested in, and plots those values. See the help for ?plotCounts. Let’s first see what the gene ID is for the CRISPLD2 gene using res %>% filter(symbol=="CRISPLD2"). Now, let’s plot the counts, where our intgroup, or “interesting group” variable is the “dex” column.

In [ ]:
plotCounts(dds, gene="ENSG00000103196", intgroup="dex")

That’s just okay. Keep looking at the help for `?plotCounts`. Notice that we could have actually returned the data instead of plotting. We could then pipe this to ggplot and make our own figure. Let’s make a boxplot.

In [ ]:
# Return the data
plotCounts(dds, gene="ENSG00000103196", intgroup="dex", returnData=TRUE)

# Plot it
plotCounts(dds, gene="ENSG00000103196", intgroup="dex", returnData=TRUE) %>% 
  ggplot(aes(dex, count)) + geom_boxplot(aes(fill=dex)) + scale_y_log10() + ggtitle("CRISPLD2")

#### MA & Volcano plots

Let’s make some commonly produced visualizations from this data. First, let’s mutate our results object to add a column called sig that evaluates to TRUE if padj<0.05, and FALSE if not, and NA if padj is also NA.

In [ ]:
# Create the new column
res <- res %>% mutate(sig=padj<0.05)

# How many of each?
res %>% 
  group_by(sig) %>% 
  summarize(n=n())

#### MA Plot
An MA plot shows the average expression on the X-axis and the log fold change on the Y-axis. A volcano plot shows the log fold change on the X-axis, and the −log10 of the p-value on the Y-axis (the more significant the p-value, the larger the −log10 of that value will be).

Here we will make an MA plot using a log10-scaled x-axis, color-coded by whether the gene is significant. What’s the deal with the gray points? Why are they missing?

**Gray points** = non‑significant genes that passed filtering and have valid statistics.

**Missing points** = genes filtered out via independent filtering (because of low mean counts / low power) or with NA in results and thus not plotted.

In [ ]:
# Add a column for “significant” (e.g., padj < 0.05)
res_df <- as.data.frame(res) %>%
  mutate(Significant = ifelse(padj < 0.05, "yes", "no"))

# Create the plot
ggplot(res_df, aes(x = baseMean, y = log2FoldChange, color = Significant)) +
  geom_point(alpha = 0.6, size = 1) +
  scale_x_log10() +                                  # log10‑scale for x‑axis
  geom_hline(yintercept = 0, linetype = "dashed", color = "gray40") +
  labs(
    title = "MA‑plot: log2FoldChange vs mean expression",
    x = "Mean of normalized counts (log10 scale)",
    y = "log2 fold change"
  ) +
  scale_color_manual(values = c("yes" = "red", "no" = "blue")) +
  theme_minimal()

The code above uses the `ggplot` function. line by line, these mean:

 
```r
aes(x = baseMean, y = log2FoldChange, color = Significant)
```
Maps mean expression to the x-axis, fold-change to the y-axis, and colors points based on significance.

```r
geom_point(alpha = 0.6, size = 1)
```
Plots each gene as a point. alpha = 0.6 makes points slightly transparent to avoid overplotting.

```r
scale_x_log10()
```
Converts the x-axis to a logarithmic scale because gene expression varies over several orders of magnitude.

```r
geom_hline(yintercept = 0, linetype = "dashed", color = "gray40")
```
Adds a horizontal line at 0 fold-change, indicating no change. Genes above this line are upregulated, below are downregulated.

```
scale_color_manual(values = c("yes" = "red", "no" = "blue"))
```
Red points = significant genes, blue points = non-significant genes.

```r
labs()
```
Adds a plot title and labels for x- and y-axes.

```r
theme_minimal()
```
Provides a clean, minimal plot background for better readability.

#### Volcano Plot
A volcano plot is a scatter plot that displays both the magnitude of change (log2FC) and statistical significance for each gene:

- **X-axis:** log2 fold change (`log2FoldChange`) → shows how much a gene is up- or down-regulated  
- **Y-axis:** -log10 of the adjusted p-value (`-log10(padj)`) → shows statistical significance  

Genes with large fold changes and low adjusted p-values stand out in the plot.

In [ ]:
#### Volcano plot
# Add a column for “significant” (e.g., padj < 0.05)
res_df <- as.data.frame(res) %>%
  mutate(Significant = ifelse(padj < 0.05, "yes", "no"))

# Volcano plot
ggplot(res_df, aes(x = log2FoldChange, y = -log10(padj), color = Significant)) +
  geom_point(alpha = 0.6, size = 1) +                       # Points
  geom_hline(yintercept = -log10(0.05), linetype = "dashed", color = "gray40") +  # Significance line
  geom_vline(xintercept = c(-1,1), linetype = "dashed", color = "gray40") +       # Fold-change cutoffs
  scale_color_manual(values = c("yes" = "red", "no" = "blue")) +
  labs(
    title = "Volcano Plot of Differential Expression",
    x = "log2 Fold Change",
    y = "-log10 adjusted P-value"
  ) +
  theme_minimal()


The code above uses the `ggplot` function. line by line, these mean:

 
```r
aes(x = log2FoldChange, y = -log10(padj), color = Significant)
```
Maps log2 fold change to the x-axis, significance (-log10(padj)) to the y-axis, and colors points based on significance.

```r
geom_point(alpha = 0.6, size = 1)
```
Plots each gene as a point, with slight transparency (alpha) to reduce overplotting.

```r
geom_hline(yintercept = -log10(0.05), linetype = "dashed", color = "gray40")
```
Adds a horizontal line at the significance threshold (adjusted p-value = 0.05).

```r
geom_vline(xintercept = c(-1,1), linetype = "dashed", color = "gray40")
```
Adds vertical lines at log2 fold change = ±1 (common fold-change cutoff).

```r
scale_color_manual(values = c("yes" = "red", "no" = "blue"))
```
Red points = significant genes (padj < 0.05), blue points = non-significant genes.

```r
labs()
```
Adds a title and axis labels.

```r
theme_minimal()
```
Provides a clean plot background for better readability.

### Transformation of the data with `vst()`
To test for differential expression, we operate on raw counts. But for other downstream analyses like heatmaps, PCA, or clustering, we need to work with transformed versions of the data, because it’s not clear how to best compute a distance metric on untransformed counts. The go-to choice might be a log transformation. But because many samples have a zero count (and log(0)=−∞, you might try using pseudocounts, i. e. y=log(n+1) or more generally, y=log(n+n0), where n represents the count values and n0 is some positive constant.

But there are other approaches that offer better theoretical justification and a rational way of choosing the parameter equivalent to n0
, and they produce transformed data on the log scale that’s normalized to library size. One is called a variance stabilizing transformation (VST), and it also removes the dependence of the variance on the mean, particularly the high variance of the log counts when the mean is low.

In [ ]:
vsdata <- vst(dds, blind=FALSE)

### PCA Plots: 

Let’s do some exploratory plotting of the data using principal components analysis on the variance stabilized data from above. Let’s use the DESeq2-provided `plotPCA` function. See the help for `?plotPCA` and notice that it also has a `returnData` option, just like `plotCounts`.

In [ ]:
plotPCA(vsdata, intgroup="dex")

Principal Components Analysis (PCA) is a dimension reduction and visualization technique that is used here to project the multivariate data vector of each sample into a two-dimensional plot, such that the spatial arrangement of the points in the plot reflects the overall data (dis)similarity between the samples. 

In essence, principal component analysis distills all the global variation between samples down to a few variables called principal components. The majority of variation between the samples can be summarized by the first principal component, which is shown on the x-axis. The second principal component summarizes the residual variation that isn’t explained by PC1. PC2 is shown on the y-axis. The percentage of the global variation explained by each principal component is given in the axis labels.

In a two-condition scenario (e.g., mutant vs WT, or treated vs control), you might expect PC1 to separate the two experimental conditions, so, for example, having all the controls on the left and all experimental samples on the right (or vice versa - the units and directionality aren’t important). 

The secondary axis may separate other aspects of the design - cell line, time point, etc. Very often, the experimental design is reflected in the PCA plot, and in this case, it is. But this kind of diagnostic can be useful for finding outliers, investigating batch effects, finding sample swaps, and other technical problems with the data. 

[This YouTube video from the Genetics Department at UNC](https://youtu.be/_UVHneBUBW0) gives a very accessible explanation of what PCA is all about in the context of a gene expression experiment, without the need for an advanced math background. Take a look.

### Heat Maps: Visualizing Top Differentially Expressed Genes

Heat maps are a common way to visualize expression patterns across samples, especially for the most differentially expressed genes.  

Here we will plot the top 50 significant genes from our DESeq2 results.

In [ ]:
# Sort DESeq2 results by adjusted p-value (padj), then p-value
# and select the top 50 genes
top_genes <- arrange(res, padj, pvalue)$row[1:50]

# Get corresponding gene symbols for labeling the rows of the heatmap
top_symbols <- arrange(res, padj, pvalue)$symbol[1:50]

# vsdata = DESeq2 variance-stabilized or rlog-transformed object
# assay(vsdata) extracts the normalized expression values
expr_mat <- assay(vsdata)[top_genes, ]
# Rows = genes, Columns = samples

# Select metadata columns of interest (treatment "dex" and "celltype")
annotation_col <- dplyr::select(metadata, dex, celltype)

# Convert to factors for proper coloring
annotation_col$dex <- factor(annotation_col$dex)
annotation_col$celltype <- factor(annotation_col$celltype)

# Remove any columns with only one unique value (cannot color them)
annotation_col <- annotation_col %>% select_if(~ n_distinct(.) > 1)


annotation_colors <- list(
  dex = c("control" = "blue", "treated" = "red"),  # colors for treatment
  celltype = c("N61311" = "green", "N052611" = "orange", 
               "N080611" = "purple", "N061011" = "brown")  # colors for cell type
)


# pheatmap requires annotation rownames to correspond to expr_mat columns
rownames(annotation_col) <- colnames(expr_mat)

#plot heatmap
pheatmap(
  expr_mat,
  scale = "row",                 # Standardize each gene (row) to mean=0, SD=1
  cluster_rows = TRUE,           # Cluster genes by similarity
  cluster_cols = TRUE,           # Cluster samples by similarity
  annotation_col = annotation_col,  # Add column annotations (treatment & cell type)
  annotation_colors = annotation_colors, # Provide colors for annotations
  labels_row = top_symbols,      # Label rows with gene symbols
  color = colorRampPalette(c("green","black","red"))(50)  # Gradient: low → high expression
)

Graphing instructions adapted from [Count-Based Differential Expression Analysis of RNA-seq Data](https://4va.github.io/biodatasci/r-rnaseq-airway.html) (Stephen Turner, 4va/biodatasci), [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/). This notebook is shared under the same license. For the same plots made from the course's Arabidopsis data, see [M4P1](../practice/M4P1_Intro_to_DE_Analysis.ipynb).